In [1]:
import sys

print(sys.version)
print(sys.executable)

3.12.16 (main, Oct 25 2026, 18:24:37) [Clang 21.0.0 (clang-2100.3.34.2)]
/Users/macbook/Documents/Python/goit-rdb-hw-03/.venv/bin/python


In [33]:
%pip install "prettytable==3.9.0"


  Using cached prettytable-3.9.0-py3-none-any.whl.metadata (26 kB)
Using cached prettytable-3.9.0-py3-none-any.whl (27 kB)
  Attempting uninstall: prettytable
    Found existing installation: prettytable 3.18.0
    Uninstalling prettytable-3.18.0:
      Successfully uninstalled prettytable-3.18.0
Note: you may need to restart the kernel to use updated packages.


In [46]:
%config SqlMagic.style = 'MSWORD_FRIENDLY'


In [47]:
import prettytable

print("PrettyTable version:", prettytable.__version__)

styles = ["DEFAULT", "PLAIN_COLUMNS", "MSWORD_FRIENDLY", "MARKDOWN", "ORGMODE"]

for style in styles:
    print(style, hasattr(prettytable, style))


PrettyTable version: 3.18.0
DEFAULT True
PLAIN_COLUMNS True
MSWORD_FRIENDLY True
MARKDOWN True
ORGMODE True


/var/folders/kk/zd8bcx4s35q_6x1xhhr72c1w0000gn/T/ipykernel_25397/4254633508.py:8: DeprecationWarning: the 'DEFAULT' constant is deprecated, use the 'TableStyle' enum instead
  print(style, hasattr(prettytable, style))
/var/folders/kk/zd8bcx4s35q_6x1xhhr72c1w0000gn/T/ipykernel_25397/4254633508.py:8: DeprecationWarning: the 'PLAIN_COLUMNS' constant is deprecated, use the 'TableStyle' enum instead
  print(style, hasattr(prettytable, style))
/var/folders/kk/zd8bcx4s35q_6x1xhhr72c1w0000gn/T/ipykernel_25397/4254633508.py:8: DeprecationWarning: the 'MSWORD_FRIENDLY' constant is deprecated, use the 'TableStyle' enum instead
  print(style, hasattr(prettytable, style))
/var/folders/kk/zd8bcx4s35q_6x1xhhr72c1w0000gn/T/ipykernel_25397/4254633508.py:8: DeprecationWarning: the 'MARKDOWN' constant is deprecated, use the 'TableStyle' enum instead
  print(style, hasattr(prettytable, style))
/var/folders/kk/zd8bcx4s35q_6x1xhhr72c1w0000gn/T/ipykernel_25397/4254633508.py:8: DeprecationWarning: the 'ORGMOD

In [48]:
%pip install --force-reinstall "prettytable==3.9.0"


  Using cached prettytable-3.9.0-py3-none-any.whl.metadata (26 kB)
  Using cached wcwidth-0.9.4-cp310-abi3-macosx_11_0_arm64.whl.metadata (23 kB)
Using cached prettytable-3.9.0-py3-none-any.whl (27 kB)
Using cached wcwidth-0.9.4-cp310-abi3-macosx_11_0_arm64.whl (618 kB)
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.9.4
    Uninstalling wcwidth-0.9.4:
      Successfully uninstalled wcwidth-0.9.4
  Attempting uninstall: prettytable━━━━━━━━━━━━━ 0/2 [wcwidth]
    Found existing installation: prettytable 3.9.00/2 [wcwidth]
    Uninstalling prettytable-3.9.0:━━━━━━━━━ 0/2 [wcwidth]
      Successfully uninstalled prettytable-3.9.0 0/2 [wcwidth]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [prettytable]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import prettytable

print(prettytable.__version__)
print("DEFAULT in dict:", "DEFAULT" in prettytable.__dict__)


3.9.0
DEFAULT in dict: True


Python — підготовка даних

Запуск pgserver, Parquet → CSV, COPY FROM STDIN

In [6]:
import pandas as pd
import pgserver

from pathlib import Path
from sqlalchemy import create_engine, text

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

# Запуск PostgreSQL
pg = pgserver.get_server(str(PROJECT_DIR / "hw3_pg"), cleanup_mode="stop")

# Підключення
engine = create_engine(pg.get_uri(), future=True)

# Перевірка
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print("PostgreSQL:", version)


PostgreSQL: PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [8]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1 + 1 AS result"))
    print(result.scalar_one())


2


In [9]:
import pyarrow.parquet as pq
from urllib.request import urlretrieve

DATA_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
)

parquet_path = DATA_DIR / "hw3_taxi_month.parquet"
csv_path = DATA_DIR / "hw3_taxi_sample.csv"

# Завантажуємо лише один місяць
if not parquet_path.exists():
    urlretrieve(DATA_URL, parquet_path)

columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
]

# Читаємо лише потрібні колонки та перші 50 000 рядків
parquet_file = pq.ParquetFile(parquet_path)

batch = next(parquet_file.iter_batches(batch_size=50_000, columns=columns))

df = batch.to_pandas()

# Зберігаємо CSV
df.to_csv(csv_path, index=False)

print("Розмір датасету:", df.shape)
print("CSV збережено:", csv_path)

display(df.head())


Розмір датасету: (50000, 11)
CSV збережено: /Users/macbook/Documents/Python/goit-rdb-hw-03/sql-homework-3/data/hw3_taxi_sample.csv


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,payment_type,fare_amount,tip_amount,total_amount
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,229,237,1,10.0,3.00,18.00
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,236,237,1,5.1,2.02,12.12
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,141,141,1,5.1,2.00,12.10
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,244,244,2,7.2,0.00,9.70
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,244,116,2,5.8,0.00,8.30


In [10]:
create_table_sql = """
DROP TABLE IF EXISTS hw3_data_clean CASCADE;
DROP TABLE IF EXISTS hw3_data_staging CASCADE;

CREATE TABLE hw3_data_staging (
    vendor_id_raw TEXT,
    pickup_datetime_raw TEXT,
    dropoff_datetime_raw TEXT,
    passenger_count_raw TEXT,
    trip_distance_raw TEXT,
    pu_location_id_raw TEXT,
    do_location_id_raw TEXT,
    payment_type_raw TEXT,
    fare_amount_raw TEXT,
    tip_amount_raw TEXT,
    total_amount_raw TEXT
);
"""

with engine.begin() as conn:
    conn.execute(text(create_table_sql))

print("Staging table created successfully!")


Staging table created successfully!


In [13]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            with cur.copy("""
                COPY hw3_data_staging (
                    vendor_id_raw,
                    pickup_datetime_raw,
                    dropoff_datetime_raw,
                    passenger_count_raw,
                    trip_distance_raw,
                    pu_location_id_raw,
                    do_location_id_raw,
                    payment_type_raw,
                    fare_amount_raw,
                    tip_amount_raw,
                    total_amount_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                )
            """) as copy:
                while chunk := file.read(1024 * 1024):
                    copy.write(chunk)

    raw_conn.commit()
    print("CSV imported successfully!")

except Exception:
    raw_conn.rollback()
    raise

finally:
    raw_conn.close()


CSV imported successfully!


In [14]:
query = """
SELECT COUNT(*) AS loaded_rows
FROM hw3_data_staging;
"""

result = pd.read_sql(query, engine)

display(result)


,loaded_rows
0,50000


Налаштовуємо часову зону New York

In [21]:
query = """
SELECT
    pickup_datetime_raw,
    pickup_datetime_raw::timestamp
        AT TIME ZONE 'America/New_York'
        AS pickup_datetime_utc
FROM hw3_data_staging
LIMIT 5;
"""

result = pd.read_sql(query, engine)

display(result)


,pickup_datetime_raw,pickup_datetime_utc
0,2025-01-01 00:18:38,2025-01-01 05:18:38+00:00
1,2025-01-01 00:32:40,2025-01-01 05:32:40+00:00
2,2025-01-01 00:44:04,2025-01-01 05:44:04+00:00
3,2025-01-01 00:14:27,2025-01-01 05:14:27+00:00
4,2025-01-01 00:21:34,2025-01-01 05:21:34+00:00


## Завдання 2. Staging і DDL

### Опис датасету

Для виконання завдання використано датасет **NYC TLC Yellow Taxi Trip Records** за січень 2025 року, який містить інформацію про поїздки жовтого таксі в Нью-Йорку.

Вибірка складається з **50 000 записів і 11 вихідних колонок**.

Початкові дані були отримані у форматі Parquet, перетворені на CSV та завантажені до staging-таблиці `hw3_data_staging`, де всі поля зберігаються як `TEXT`.

Для подальшого аналізу створено фінальну таблицю `hw3_taxi_trips` із відповідними типами PostgreSQL та обмеженнями цілісності.

### Семантика колонок, типи даних і правила очищення

| Колонка | Семантика | Джерельний формат | Тип PostgreSQL | Правила очищення та обґрунтування |
|---|---|---|---|---|
| `trip_id` | Унікальний ідентифікатор поїздки | Відсутній у джерелі | `BIGINT IDENTITY PRIMARY KEY` | Генерується автоматично, оскільки датасет не містить унікального ключа для кожної поїздки. |
| `vendor_id` | Код компанії-постачальника даних таксометра | Числовий код, у staging — `TEXT` | `SMALLINT` | Видалення пробілів через `TRIM()`, порожні значення перетворюються на `NULL`. `SMALLINT` достатній для невеликих числових ідентифікаторів. |
| `pickup_at` | Дата та час початку поїздки | Локальний `datetime` без часової зони, у staging — `TEXT` | `TIMESTAMPTZ NOT NULL` | Перетворення на `TIMESTAMP` і локалізація через `AT TIME ZONE 'America/New_York'`. Записи без дати початку виключаються. |
| `dropoff_at` | Дата та час завершення поїздки | Локальний `datetime` без часової зони, у staging — `TEXT` | `TIMESTAMPTZ NOT NULL` | Локалізація New York. Записи з відсутньою датою або завершенням раніше початку поїздки виключаються. |
| `passenger_count` | Кількість пасажирів у таксі | Числове значення, у staging — `TEXT` | `SMALLINT` | Порожні значення → `NULL`. Від'ємні значення виключаються через `CHECK (passenger_count >= 0)`. |
| `trip_distance` | Відстань поїздки в милях | Десяткове число, у staging — `TEXT` | `NUMERIC(10,2)` | Порожні значення → `NULL`, від'ємні відстані виключаються. Тип забезпечує зберігання відстані з точністю до сотих милі. |
| `pu_location_id` | Ідентифікатор географічної зони посадки | Числовий код, у staging — `TEXT` | `INTEGER` | Видалення пробілів, порожні значення → `NULL`. `INTEGER` використовується для зберігання числового ідентифікатора зони. |
| `do_location_id` | Ідентифікатор географічної зони висадки | Числовий код, у staging — `TEXT` | `INTEGER` | Видалення пробілів, порожні значення → `NULL`. |
| `payment_type` | Код способу оплати (картка, готівка тощо) | Категоріальний числовий код, у staging — `TEXT` | `SMALLINT` | Видалення пробілів, порожні значення → `NULL`. Це категоріальна ознака, а не грошове значення. |
| `fare_amount` | Вартість поїздки за тарифом | Десяткове число, у staging — `TEXT` | `NUMERIC(10,2)` | Порожні значення → `NULL`. Тип `NUMERIC(10,2)` забезпечує точність до центів і не використовує наближені обчислення з плаваючою комою. |
| `tip_amount` | Сума чайових | Десяткове число, у staging — `TEXT` | `NUMERIC(10,2)` | Порожні значення → `NULL`, сума зберігається з точністю до центів. |
| `total_amount` | Загальна сума оплати за поїздку | Десяткове число, у staging — `TEXT` | `NUMERIC(10,2)` | Порожні значення → `NULL`, зберігається точність до центів. |

### Правила очищення даних

Під час перенесення даних зі staging-таблиці до фінальної використано такі правила:

1. **Обробка порожніх значень:** `TRIM()` видаляє пробіли на початку та в кінці тексту, а `NULLIF(..., '')` замінює порожні рядки на `NULL`.
2. **Перетворення типів:** текстові значення приводяться до `SMALLINT`, `INTEGER`, `NUMERIC(10,2)` і `TIMESTAMPTZ`.
3. **Локалізація часу:** дати початку та завершення поїздок інтерпретуються як локальний час Нью-Йорка через `AT TIME ZONE 'America/New_York'`.
4. **Перевірка дат:** виключаються записи без часу початку або завершення, а також записи, де `dropoff_at < pickup_at`.
5. **Перевірка числових значень:** виключаються записи з від'ємною відстанню або від'ємною кількістю пасажирів. `NULL` у цих двох полях допускається.
6. **Збереження грошових значень:** використовується `NUMERIC(10,2)` для точного зберігання сум у доларах США. Від'ємні грошові значення не виключаються автоматично, оскільки можуть відображати коригування платежів.

### Обмеження цілісності

У фінальній таблиці застосовано:

- `PRIMARY KEY` — забезпечує унікальність `trip_id`.
- `NOT NULL` — гарантує наявність часу початку та завершення поїздки.
- `CHECK (passenger_count >= 0)` — забороняє від'ємну кількість пасажирів.
- `CHECK (trip_distance >= 0)` — забороняє від'ємну відстань.
- `CHECK (dropoff_at >= pickup_at)` — перевіряє правильний хронологічний порядок подій.

Типи `BOOLEAN` та `DATE` не використовуються, оскільки серед вибраних полів немає окремих двійкових ознак або календарних дат без часу.

### Результат

Після очищення та перетворення типів із таблиці `hw3_data_staging` до `hw3_taxi_trips` успішно перенесено **50 000 із 50 000 записів (100%)**.

Усі записи пройшли застосовані перевірки, а фінальна таблиця містить типізовані дані, готові до подальшого аналізу.

In [7]:
%reload_ext sql
%config SqlMagic.style = 'DEFAULT'

db_url = pg.get_uri()
%sql $db_url


In [8]:
%%sql

SELECT version();

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


version
"PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit"


In [44]:
%%sql

DROP TABLE IF EXISTS hw3_taxi_trips CASCADE;

CREATE TABLE hw3_taxi_trips (
    trip_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,

    vendor_id SMALLINT,

    pickup_at TIMESTAMPTZ NOT NULL,
    dropoff_at TIMESTAMPTZ NOT NULL,

    passenger_count SMALLINT
        CHECK (passenger_count >= 0),

    trip_distance NUMERIC(10,2)
        CHECK (trip_distance >= 0),

    pu_location_id INTEGER,
    do_location_id INTEGER,

    payment_type SMALLINT,

    fare_amount NUMERIC(10,2),
    tip_amount NUMERIC(10,2),
    total_amount NUMERIC(10,2),

    CHECK (dropoff_at >= pickup_at)
);

   postgresql+psycopg://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
Done.
Done.


[]

In [9]:
%%sql

SELECT COUNT(*) AS total_rows
FROM hw3_taxi_trips;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


total_rows
0


Перенесення очищених даних зі staging-таблиці до фінальної через INSERT INTO ... SELECT

In [10]:
%%sql

SELECT COUNT(*) AS staging_rows
FROM hw3_data_staging;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


staging_rows
50000


Перенесимо дані до фінальної таблиці

In [11]:
%%sql

WITH parsed AS (
    SELECT
        NULLIF(TRIM(vendor_id_raw), '')::SMALLINT AS vendor_id,

        NULLIF(TRIM(pickup_datetime_raw), '')::TIMESTAMP
            AT TIME ZONE 'America/New_York' AS pickup_at,

        NULLIF(TRIM(dropoff_datetime_raw), '')::TIMESTAMP
            AT TIME ZONE 'America/New_York' AS dropoff_at,

        NULLIF(TRIM(passenger_count_raw), '')::NUMERIC::SMALLINT
            AS passenger_count,

        NULLIF(TRIM(trip_distance_raw), '')::NUMERIC(10,2)
            AS trip_distance,

        NULLIF(TRIM(pu_location_id_raw), '')::INTEGER
            AS pu_location_id,

        NULLIF(TRIM(do_location_id_raw), '')::INTEGER
            AS do_location_id,

        NULLIF(TRIM(payment_type_raw), '')::SMALLINT
            AS payment_type,

        NULLIF(TRIM(fare_amount_raw), '')::NUMERIC(10,2)
            AS fare_amount,

        NULLIF(TRIM(tip_amount_raw), '')::NUMERIC(10,2)
            AS tip_amount,

        NULLIF(TRIM(total_amount_raw), '')::NUMERIC(10,2)
            AS total_amount

    FROM hw3_data_staging
)

INSERT INTO hw3_taxi_trips (
    vendor_id,
    pickup_at,
    dropoff_at,
    passenger_count,
    trip_distance,
    pu_location_id,
    do_location_id,
    payment_type,
    fare_amount,
    tip_amount,
    total_amount
)

SELECT
    vendor_id,
    pickup_at,
    dropoff_at,
    passenger_count,
    trip_distance,
    pu_location_id,
    do_location_id,
    payment_type,
    fare_amount,
    tip_amount,
    total_amount

FROM parsed

WHERE pickup_at IS NOT NULL
  AND dropoff_at IS NOT NULL
  AND dropoff_at >= pickup_at
  AND (trip_distance IS NULL OR trip_distance >= 0)
  AND (passenger_count IS NULL OR passenger_count >= 0);

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
50000 rows affected.


[]

In [12]:
%%sql

SELECT
    (SELECT COUNT(*) FROM hw3_data_staging) AS staging_rows,
    (SELECT COUNT(*) FROM hw3_taxi_trips) AS final_rows;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


staging_rows,final_rows
50000,50000


In [13]:
%%sql

SELECT *
FROM hw3_taxi_trips
LIMIT 5;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
5 rows affected.


trip_id,vendor_id,pickup_at,dropoff_at,passenger_count,trip_distance,pu_location_id,do_location_id,payment_type,fare_amount,tip_amount,total_amount
1,1,2025-01-01 06:18:38+01:00,2025-01-01 06:26:59+01:00,1,1.60,229,237,1,10.00,3.00,18.00
2,1,2025-01-01 06:32:40+01:00,2025-01-01 06:35:13+01:00,1,0.50,236,237,1,5.10,2.02,12.12
3,1,2025-01-01 06:44:04+01:00,2025-01-01 06:46:01+01:00,1,0.60,141,141,1,5.10,2.00,12.10
4,2,2025-01-01 06:14:27+01:00,2025-01-01 06:20:01+01:00,3,0.52,244,244,2,7.20,0.00,9.70
5,2,2025-01-01 06:21:34+01:00,2025-01-01 06:25:06+01:00,3,0.66,244,116,2,5.80,0.00,8.30


Перевірка якості очищених даних

In [14]:
%%sql

SELECT
    COUNT(*) AS total_trips,

    COUNT(*) FILTER (
        WHERE dropoff_at < pickup_at
    ) AS invalid_time,

    COUNT(*) FILTER (
        WHERE trip_distance < 0
    ) AS negative_distance,

    COUNT(*) FILTER (
        WHERE passenger_count < 0
    ) AS negative_passengers,

    COUNT(*) FILTER (
        WHERE pickup_at IS NULL OR dropoff_at IS NULL
    ) AS missing_dates

FROM hw3_taxi_trips;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


total_trips,invalid_time,negative_distance,negative_passengers,missing_dates
50000,0,0,0,0


Перевірка типів колонок

In [15]:
%%sql

SELECT
    column_name,
    data_type,
    is_nullable
FROM information_schema.columns
WHERE table_name = 'hw3_taxi_trips'
ORDER BY ordinal_position;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
12 rows affected.


column_name,data_type,is_nullable
trip_id,bigint,NO
vendor_id,smallint,YES
pickup_at,timestamp with time zone,NO
dropoff_at,timestamp with time zone,NO
passenger_count,smallint,YES
trip_distance,numeric,YES
pu_location_id,integer,YES
do_location_id,integer,YES
payment_type,smallint,YES
fare_amount,numeric,YES


3.1. Контроль кількості рядків

In [16]:
%%sql

SELECT
    'staging' AS layer,
    COUNT(*) AS n_rows
FROM hw3_data_staging

UNION ALL

SELECT
    'clean' AS layer,
    COUNT(*) AS n_rows
FROM hw3_taxi_trips;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
2 rows affected.


layer,n_rows
clean,50000
staging,50000


**Висновок:** Кількість рядків у staging-таблиці та фінальній таблиці однакова — 50 000 записів.

Під час перенесення даних застосовувалися правила фільтрації: обов'язкова наявність часу початку та завершення поїздки, завершення не раніше початку, невід'ємна відстань і невід'ємна кількість пасажирів (якщо ці значення не `NULL`).

Усі 50 000 записів пройшли зазначені перевірки, тому жоден рядок не був відфільтрований.

3.2. Перевірка дублікатів

In [17]:
%%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT(DISTINCT (
        vendor_id,
        pickup_at,
        dropoff_at,
        pu_location_id,
        do_location_id,
        trip_distance,
        fare_amount,
        total_amount
    )) AS distinct_business_keys,

    COUNT(*) - COUNT(DISTINCT (
        vendor_id,
        pickup_at,
        dropoff_at,
        pu_location_id,
        do_location_id,
        trip_distance,
        fare_amount,
        total_amount
    )) AS duplicate_rows

FROM hw3_taxi_trips;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


total_rows,distinct_business_keys,duplicate_rows
50000,50000,0


**Висновок:** Для перевірки дублікатів використано складений business key, що включає ідентифікатор перевізника, час початку й завершення поїздки, локації посадки та висадки, відстань і грошові показники.

Оскільки датасет NYC Taxi не містить гарантованого унікального природного ключа для кожної поїздки, знайдені збіги розглядаються як потенційні дублікати, а не як однозначно помилкові записи.

**Результат перевірки:** 0.

3.3. Null-rate і спеціальні маркери

In [18]:
%%sql

SELECT
    COUNT(*) AS total_rows,

    ROUND(
        COUNT(*) FILTER (WHERE vendor_id IS NULL) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS pct_null_vendor,

    ROUND(
        COUNT(*) FILTER (WHERE pickup_at IS NULL) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS pct_null_pickup,

    ROUND(
        COUNT(*) FILTER (WHERE dropoff_at IS NULL) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS pct_null_dropoff,

    ROUND(
        COUNT(*) FILTER (WHERE passenger_count IS NULL) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS pct_null_passengers,

    ROUND(
        COUNT(*) FILTER (WHERE trip_distance IS NULL) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS pct_null_distance,

    ROUND(
        COUNT(*) FILTER (WHERE payment_type IS NULL) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS pct_null_payment,

    ROUND(
        COUNT(*) FILTER (WHERE fare_amount IS NULL) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS pct_null_fare,

    ROUND(
        COUNT(*) FILTER (WHERE tip_amount IS NULL) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS pct_null_tip,

    ROUND(
        COUNT(*) FILTER (WHERE total_amount IS NULL) * 100.0
        / NULLIF(COUNT(*), 0), 2
    ) AS pct_null_total

FROM hw3_taxi_trips;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


total_rows,pct_null_vendor,pct_null_pickup,pct_null_dropoff,pct_null_passengers,pct_null_distance,pct_null_payment,pct_null_fare,pct_null_tip,pct_null_total
50000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


SQL-запит 2. Спеціальні маркери у staging

In [19]:
%%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE LOWER(TRIM(vendor_id_raw)) IN ('?', 'unknown', '')
    ) AS missing_vendor_markers,

    COUNT(*) FILTER (
        WHERE LOWER(TRIM(passenger_count_raw)) IN ('?', 'unknown', '')
    ) AS missing_passenger_markers,

    COUNT(*) FILTER (
        WHERE LOWER(TRIM(trip_distance_raw)) IN ('?', 'unknown', '')
    ) AS missing_distance_markers,

    COUNT(*) FILTER (
        WHERE LOWER(TRIM(payment_type_raw)) IN ('?', 'unknown', '')
    ) AS missing_payment_markers,

    COUNT(*) FILTER (
        WHERE LOWER(TRIM(fare_amount_raw)) IN ('?', 'unknown', '')
    ) AS missing_fare_markers,

    COUNT(*) FILTER (
        WHERE LOWER(TRIM(total_amount_raw)) IN ('?', 'unknown', '')
    ) AS missing_total_markers

FROM hw3_data_staging;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


total_rows,missing_vendor_markers,missing_passenger_markers,missing_distance_markers,missing_payment_markers,missing_fare_markers,missing_total_markers
50000,0,0,0,0,0,0


SQL-запит 3. Перевірка спеціальних категорій

In [20]:
%%sql

SELECT
    payment_type,
    COUNT(*) AS trip_count

FROM hw3_taxi_trips

GROUP BY payment_type

ORDER BY trip_count DESC;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
4 rows affected.


payment_type,trip_count
1,37946
2,10016
4,1563
3,475


Висновок: Проведено аналіз частки пропущених значень у дев'яти важливих колонках фінальної таблиці hw3_taxi_trips.

Додатково перевірено staging-таблицю на наявність спеціальних маркерів пропусків (?, unknown, порожній рядок), а також розподіл категорій способу оплати.

Результати перевірки: 0.

4.1. Розподіл категоріальної колонки

Для аналізу оберемо payment_type — спосіб оплати поїздки.

Це категоріальна ознака, яка показує, як пасажири оплачували поїздки.

In [21]:
%%sql

SELECT
    payment_type,
    COUNT(*) AS n_rows,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS pct_rows
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY n_rows DESC, payment_type
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
4 rows affected.


payment_type,n_rows,pct_rows
1,37946,75.89
2,10016,20.03
4,1563,3.13
3,475,0.95


### 4.1. Розподіл способів оплати

Для аналізу категоріальних даних обрано колонку `payment_type`, яка визначає спосіб оплати поїздки.

За результатами SQL-запиту найпоширенішою категорією є Credit card, яка становить **75,89%** усіх поїздок. Друга за поширеністю категорія — **[Cash]**, її частка становить **[20]%**.

Розподіл демонструє наявність дисбалансу між способами оплати. 

4.2. Базова статистика числової колонки

In [23]:
%%sql

SELECT
    COUNT(trip_distance) AS n_non_null,
    MIN(trip_distance) AS min_value,
    MAX(trip_distance) AS max_value,
    ROUND(AVG(trip_distance), 2) AS avg_value,
    ROUND(STDDEV_SAMP(trip_distance), 2) AS sd_value
FROM hw3_taxi_trips;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


n_non_null,min_value,max_value,avg_value,sd_value
50000,0.00,133.30,3.68,4.83


### 4.2. Базова статистика числової колонки

Для аналізу обрано колонку `trip_distance`, яка містить відстань поїздок таксі в милях.

За результатами аналізу 50 000 записів отримано такі показники:

- Мінімальна відстань — **0,00 миль**.
- Максимальна відстань — **133,30 миль**.
- Середня відстань — **3,68 милі**.
- Стандартне відхилення — **4,83 милі**.

У колонці `trip_distance` відсутні пропущені значення.

Стандартне відхилення перевищує середнє значення, що свідчить про значний розкид відстаней. Максимальна відстань (133,30 миль) суттєво перевищує середню (3,68 милі), тому в даних можуть бути нетипово довгі поїздки.

Також виявлено мінімальне значення 0,00 миль, яке потребує додаткової перевірки.

**Висновок:** розподіл відстаней може містити потенційні аномалії, але автоматично виключати великі або нульові значення не варто без аналізу контексту поїздок.

In [24]:
%%sql

SELECT
    COUNT(*) FILTER (
        WHERE trip_distance = 0
    ) AS zero_distance_trips,

    COUNT(*) FILTER (
        WHERE trip_distance > 50
    ) AS trips_over_50_miles,

    COUNT(*) FILTER (
        WHERE trip_distance > 100
    ) AS trips_over_100_miles

FROM hw3_taxi_trips;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


zero_distance_trips,trips_over_50_miles,trips_over_100_miles
996,15,5


4.3. Перевірка потенційних аномалій (outlier-check)

In [25]:
%%sql

SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE fare_amount < 0
    ) AS negative_fare,

    COUNT(*) FILTER (
        WHERE tip_amount < 0
    ) AS negative_tip,

    COUNT(*) FILTER (
        WHERE total_amount > 10000
    ) AS extremely_high_total,

    COUNT(*) FILTER (
        WHERE fare_amount < 0
           OR tip_amount < 0
           OR total_amount > 10000
    ) AS total_anomalous_rows

FROM hw3_taxi_trips;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


total_rows,negative_fare,negative_tip,extremely_high_total,total_anomalous_rows
50000,1373,3,0,1373


Перегляд аномальних записів

In [26]:
%%sql

SELECT
    trip_id,
    pickup_at,
    dropoff_at,
    trip_distance,
    payment_type,
    fare_amount,
    tip_amount,
    total_amount

FROM hw3_taxi_trips

WHERE fare_amount < 0
   OR tip_amount < 0
   OR total_amount > 10000

ORDER BY pickup_at

LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
20 rows affected.


trip_id,pickup_at,dropoff_at,trip_distance,payment_type,fare_amount,tip_amount,total_amount
5519,2025-01-01 06:00:07+01:00,2025-01-01 06:06:31+01:00,0.93,3,-7.90,0.00,-12.90
18,2025-01-01 06:01:41+01:00,2025-01-01 06:07:14+01:00,0.71,2,-7.20,3.66,-8.54
3351,2025-01-01 06:02:03+01:00,2025-01-01 06:09:42+01:00,0.40,4,-7.90,0.00,-12.90
1806,2025-01-01 06:02:08+01:00,2025-01-01 06:13:45+01:00,2.05,4,-13.50,0.00,-18.50
1132,2025-01-01 06:02:20+01:00,2025-01-01 06:03:34+01:00,0.12,2,-3.70,0.00,-8.70
5858,2025-01-01 06:02:49+01:00,2025-01-01 06:06:00+01:00,0.48,4,-5.10,0.00,-7.60
3724,2025-01-01 06:03:23+01:00,2025-01-01 06:12:20+01:00,1.82,4,-10.70,0.00,-15.70
5786,2025-01-01 06:03:24+01:00,2025-01-01 06:12:22+01:00,4.43,4,-19.10,0.00,-23.35
5215,2025-01-01 06:04:17+01:00,2025-01-01 06:06:03+01:00,0.04,3,-3.70,0.00,-8.70
5737,2025-01-01 06:04:38+01:00,2025-01-01 06:05:19+01:00,0.00,3,-12.40,0.00,-15.90


### 4.3. Перевірка потенційних аномалій (Outlier-check)

Для виявлення потенційних аномалій проведено перевірку грошових показників 50 000 поїздок NYC Yellow Taxi.

**Результати перевірки:**

- Від'ємна вартість поїздки (`fare_amount < 0`) — **1 373 записи**.
- Від'ємні чайові (`tip_amount < 0`) — **3 записи**.
- Надзвичайно висока сума (`total_amount > 10000`) — **0 записів**.
- Загальна кількість записів із потенційними аномаліями — **1 373 (2,75%)**.

Аналіз перших 20 аномальних записів показав, що від'ємні суми часто зустрічаються для `payment_type = 3` (No charge) та `payment_type = 4` (Dispute). Це може свідчити про скасування поїздок, повернення коштів або коригування транзакцій.

**Рекомендовані дії:** не видаляти такі записи автоматично, оскільки вони можуть відображати реальні фінансові операції. Доцільно додати прапорець `is_anomaly` для їх ідентифікації та за потреби перевірити першоджерело. Для аналізу звичайних оплачуваних поїздок такі записи можна виключати за окремим правилом.

**Висновок:** у датасеті виявлено 2,75% потенційно аномальних записів. Вони потребують додаткової перевірки, проте не обов'язково є помилками даних.

5.1. Які способи оплати найпопулярніші?

In [27]:
%%sql

SELECT
    payment_type,
    COUNT(*) AS total_trips,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM hw3_taxi_trips
GROUP BY payment_type
ORDER BY total_trips DESC
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
4 rows affected.


payment_type,total_trips,percentage
1,37946,75.89
2,10016,20.03
4,1563,3.13
3,475,0.95


Найпопулярнішим способом оплати є банківська картка (`payment_type = 1`) — 37 946 поїздок (75,89%). На другому місці готівка (`payment_type = 2`) — 10 016 поїздок (20,03%). Категорії `Dispute` (3,13%) та `No charge` (0,95%) зустрічаються значно рідше. Отже, спостерігається суттєвий дисбаланс: понад 75% усіх поїздок оплачуються картко

5.2. У які години найбільше поїздок?

In [28]:
%%sql

SELECT
    EXTRACT(
        HOUR FROM pickup_at AT TIME ZONE 'America/New_York'
    )::INTEGER AS pickup_hour,
    COUNT(*) AS total_trips
FROM hw3_taxi_trips
GROUP BY pickup_hour
ORDER BY total_trips DESC, pickup_hour
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
10 rows affected.


pickup_hour,total_trips
0,5939
1,5783
2,4870
15,4513
14,4489
13,4143
12,3522
3,3405
11,2763
4,2054


Найбільшу кількість поїздок зафіксовано о 00:00 (5 939), 01:00 (5 783) та 02:00 (4 870) за місцевим часом Нью-Йорка. Також висока активність спостерігається в денні години: о 15:00 — 4 513 поїздок, о 14:00 — 4 489. У досліджуваній вибірці найбільша кількість поїздок припадає на нічні години, хоча вдень також спостерігається значний попит.

5.3. Яка частка поїздок має невідому кількість пасажирів?

In [29]:
%%sql

SELECT
    COUNT(*) AS total_trips,
    COUNT(*) FILTER (
        WHERE passenger_count IS NULL
    ) AS missing_passenger_count,
    ROUND(
        COUNT(*) FILTER (
            WHERE passenger_count IS NULL
        ) * 100.0 / NULLIF(COUNT(*), 0),
        2
    ) AS missing_percentage
FROM hw3_taxi_trips;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
1 rows affected.


total_trips,missing_passenger_count,missing_percentage
50000,0,0.00


У вибірці з 50 000 поїздок не виявлено жодного пропущеного значення (`NULL`) у колонці `passenger_count`. Частка пропусків становить 0%, що свідчить про повноту цього поля в досліджуваній вибірці. Водночас відсутність `NULL` не гарантує коректності всіх значень, тому додатково варто перевірити наявність нульової або нетипово великої кількості пасажирів.

5.4. Які поїздки мають підозрілі значення?

In [30]:
%%sql

SELECT
    trip_id,
    pickup_at,
    trip_distance,
    fare_amount,
    tip_amount,
    total_amount,
    payment_type
FROM hw3_taxi_trips
WHERE
    (
        fare_amount < 0
        OR trip_distance = 0
        OR total_amount > 10000
    )
    AND NOT (payment_type IS NULL)
ORDER BY pickup_at
LIMIT 20;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
20 rows affected.


trip_id,pickup_at,trip_distance,fare_amount,tip_amount,total_amount,payment_type
5519,2025-01-01 06:00:07+01:00,0.93,-7.90,0.00,-12.90,3
4534,2025-01-01 06:01:19+01:00,0.00,0.00,0.00,-2.50,2
4535,2025-01-01 06:01:19+01:00,0.00,0.00,0.00,2.50,2
18,2025-01-01 06:01:41+01:00,0.71,-7.20,3.66,-8.54,2
3351,2025-01-01 06:02:03+01:00,0.40,-7.90,0.00,-12.90,4
5477,2025-01-01 06:02:07+01:00,0.00,13.00,4.12,20.62,1
1806,2025-01-01 06:02:08+01:00,2.05,-13.50,0.00,-18.50,4
1132,2025-01-01 06:02:20+01:00,0.12,-3.70,0.00,-8.70,2
5858,2025-01-01 06:02:49+01:00,0.48,-5.10,0.00,-7.60,4
5572,2025-01-01 06:02:53+01:00,0.00,83.00,0.00,85.75,4


Запит виявив поїздки з від'ємною тарифною вартістю (`fare_amount < 0`) та нульовою відстанню (`trip_distance = 0`). Серед перших 20 результатів зустрічаються від'ємні суми, наприклад −109,80 долара, а також поїздки з нульовою відстанню та додатною вартістю, зокрема 83 долари. Від'ємні суми часто пов'язані з категоріями оплати `Dispute` та `No charge`, що може свідчити про коригування або повернення платежів. Такі записи потребують додаткової перевірки, але не повинні автоматично видалятися з датасету.

5.5. Яка середня вартість поїздок залежно від способу оплати?

In [31]:
%%sql

SELECT
    payment_type,
    COUNT(*) AS total_trips,
    ROUND(AVG(total_amount), 2) AS avg_total_amount,
    MIN(total_amount) AS min_total_amount,
    MAX(total_amount) AS max_total_amount
FROM hw3_taxi_trips
WHERE payment_type IN (1, 2)
  AND total_amount >= 0
GROUP BY payment_type
ORDER BY avg_total_amount DESC;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
2 rows affected.


payment_type,total_trips,avg_total_amount,min_total_amount,max_total_amount
1,37946,29.65,3.60,478.50
2,9680,26.11,0.00,794.82


Середня вартість поїздок, оплачених банківською карткою (`payment_type = 1`), становить **29,65 долара**, тоді як готівкою (`payment_type = 2`) — **26,11 долара**. Отже, середня сума поїздки при оплаті карткою на **3,54 долара** вища, ніж при оплаті готівкою. Максимальна вартість поїздки при оплаті готівкою становить 794,82 долара, що перевищує максимальну суму при оплаті карткою (478,50 долара). Це свідчить про відмінності у вартості поїздок залежно від способу оплати, але не доводить причинно-наслідкового зв'язку.

5.6. Які поїздки мають найбільшу відстань?

In [32]:
%%sql

SELECT
    trip_id,
    pickup_at,
    dropoff_at,
    trip_distance,
    fare_amount,
    total_amount
FROM hw3_taxi_trips
WHERE trip_distance BETWEEN 0 AND 200
ORDER BY trip_distance DESC
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
10 rows affected.


trip_id,pickup_at,dropoff_at,trip_distance,fare_amount,total_amount
9221,2025-01-01 07:04:01+01:00,2025-01-01 09:33:22+01:00,133.30,350.00,380.88
29706,2025-01-01 17:32:19+01:00,2025-01-01 18:01:14+01:00,122.77,63.20,64.70
34811,2025-01-01 18:53:37+01:00,2025-01-01 21:25:57+01:00,119.66,773.00,794.82
23941,2025-01-01 13:41:21+01:00,2025-01-01 16:05:39+01:00,116.90,350.00,388.01
25185,2025-01-01 14:44:11+01:00,2025-01-01 16:12:44+01:00,105.24,300.00,361.20
21397,2025-01-01 10:51:56+01:00,2025-01-01 13:03:12+01:00,97.58,372.00,386.38
10047,2025-01-01 07:05:24+01:00,2025-01-01 08:35:52+01:00,97.42,310.00,311.00
28090,2025-01-01 16:05:04+01:00,2025-01-01 17:54:48+01:00,97.34,500.00,516.13
23685,2025-01-01 12:02:52+01:00,2025-01-01 12:29:25+01:00,94.88,70.00,94.13
28888,2025-01-01 16:38:17+01:00,2025-01-01 23:13:12+01:00,84.42,484.60,525.36


Найдовша поїздка у вибірці має відстань 133,30 милі та загальну вартість 380,88 долара. Друга за довжиною поїздка становить 122,77 милі, але її вартість значно нижча — 64,70 долара. Максимальна відстань приблизно у 36 разів перевищує середню (3,68 милі), що свідчить про наявність нетипово довгих поїздок. Такі записи варто додатково перевірити, однак велика відстань сама по собі не означає помилку в даних.

5.7. Які унікальні коди постачальників присутні в датасеті?

In [33]:
%%sql

SELECT DISTINCT
    vendor_id
FROM hw3_taxi_trips
WHERE vendor_id IS NOT NULL
ORDER BY vendor_id;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
3 rows affected.


vendor_id
1
2
7


У вибірці NYC Yellow Taxi виявлено 3 унікальні коди постачальників даних: 1, 2 та 7. Це означає, що записи надходять від трьох категорій постачальників, які ідентифікуються полем vendor_id. Ці коди не позначають окремих водіїв чи автомобілі, а використовуються для ідентифікації постачальників даних про поїздки.

5.8. Чи відрізняються записи з нульовою відстанню від інших?

In [34]:
%%sql

SELECT
    CASE
        WHEN trip_distance = 0 THEN 'Zero distance'
        ELSE 'Positive distance'
    END AS distance_category,
    COUNT(*) AS total_trips,
    ROUND(AVG(total_amount), 2) AS avg_total_amount
FROM hw3_taxi_trips
WHERE trip_distance IS NOT NULL
GROUP BY distance_category
ORDER BY total_trips DESC;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
2 rows affected.


distance_category,total_trips,avg_total_amount
Positive distance,49004,27.06
Zero distance,996,34.93


Із 50 000 поїздок 996 (1,99%) мають нульову відстань, тоді як 49 004 (98,01%) — додатну. Середня загальна вартість поїздок із нульовою відстанню становить 34,93 долара, що на 7,87 долара більше, ніж для поїздок із додатною відстанню (27,06 долара). Це свідчить про наявність нетипових записів, які можуть бути пов'язані з фіксованими тарифами, особливостями обліку або помилками даних. Такі поїздки варто додатково перевірити, але не видаляти автоматично.

5.9. Чи є відмінності між тарифною та загальною сумою?

In [35]:
%%sql

SELECT
    trip_id,
    fare_amount,
    tip_amount,
    total_amount,
    ROUND(
        total_amount - COALESCE(fare_amount, 0), 2
    ) AS additional_charges
FROM hw3_taxi_trips
WHERE tip_amount IS NOT DISTINCT FROM 0
  AND total_amount IS NOT NULL
  AND fare_amount IS NOT NULL
ORDER BY additional_charges DESC
LIMIT 10;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
10 rows affected.


trip_id,fare_amount,tip_amount,total_amount,additional_charges
28888,484.60,0.00,525.36,40.76
27697,184.00,0.00,223.75,39.75
43814,92.30,0.00,131.68,39.38
17342,86.70,0.00,125.46,38.76
23941,350.00,0.00,388.01,38.01
33956,141.30,0.00,178.06,36.76
38920,68.10,0.00,103.48,35.38
34931,98.60,0.00,133.30,34.70
38564,113.60,0.00,147.98,34.38
31230,119.90,0.00,154.28,34.38


Аналіз показав, що навіть за відсутності чайових (tip_amount = 0) загальна сума поїздки може суттєво перевищувати базовий тариф (fare_amount). Серед 10 записів із найбільшою різницею додаткові нарахування становлять від 34,38 до 40,76 долара. Найбільшу різницю зафіксовано для поїздки trip_id = 28888: базовий тариф становить 484,60 долара, а загальна сума — 525,36 долара. Це свідчить про наявність додаткових платежів, які можуть включати податки, збори та інші доплати, навіть якщо пасажир не залишив чайових.

5.10. Які способи оплати мають від'ємні суми?

In [36]:
%%sql

WITH payment_labels AS (
    SELECT
        payment_type,
        fare_amount,
        CASE payment_type
            WHEN 1 THEN 'Credit card'
            WHEN 2 THEN 'Cash'
            WHEN 3 THEN 'No charge'
            WHEN 4 THEN 'Dispute'
            WHEN 5 THEN 'Unknown'
            ELSE 'Other'
        END AS payment_label
    FROM hw3_taxi_trips
)
SELECT
    payment_label,
    COUNT(*) AS negative_fare_trips
FROM payment_labels
WHERE fare_amount < 0
  AND (
      payment_label ILIKE '%charge%'
      OR payment_label ILIKE '%dispute%'
  )
GROUP BY payment_label
ORDER BY negative_fare_trips DESC;

 * postgresql://postgres:***@/postgres?host=%2FUsers%2Fmacbook%2FDocuments%2FPython%2Fgoit-rdb-hw-03%2Fsql-homework-3%2Fhw3_pg
2 rows affected.


payment_label,negative_fare_trips
Dispute,874
No charge,174


Аналіз показав, що серед вибраних категорій від'ємні суми тарифів найчастіше зустрічаються для способу оплати Dispute (спірні транзакції) — 874 поїздки. Для категорії No charge (без оплати) виявлено 174 поїздки з від'ємними сумами. Загалом у цих двох категоріях зафіксовано 1048 таких записів, з яких приблизно 83,4% припадає на Dispute. Це може свідчити про повернення коштів або коригування платежів, тому такі записи варто перевіряти окремо, а не автоматично вважати помилками.

## Завдання 6. Reflection

Під час аналізу датасету NYC Taxi я виявила кілька проблем якості даних. Хоча перевірка не показала пропусків у кількості пасажирів, були знайдені від’ємні суми тарифів, нульові відстані та незвично великі значення. Частина таких записів може бути пов’язана з поверненням коштів або скасуванням поїздок, тому я вирішила не видаляти їх автоматично. Натомість доцільно позначати підозрілі записи для подальшого аналізу.

Для машинного навчання я використала б час початку поїздки (`pickup_at`), місце посадки (`pu_location_id`), місце висадки (`do_location_id`) та кількість пасажирів (`passenger_count`). Ці ознаки можуть допомогти прогнозувати вартість поїздки. Технічний ідентифікатор `trip_id` я б виключила, оскільки він не містить корисної інформації для прогнозування.

Важливо також враховувати ризик **data leakage**. Наприклад, якщо мета — прогнозувати вартість поїздки до її початку, не можна використовувати `total_amount`, `tip_amount` або фактичний час завершення `dropoff_at`, оскільки ці значення стають відомими лише після поїздки. Фактичну відстань також потрібно замінити на попередньо розраховану.

Для покращення data pipeline я запропонувала б автоматизувати перевірки типів даних, допустимих діапазонів, дублікатів і пропущених значень. Також варто додати логування помилок та окремі прапорці для аномальних транзакцій.

Найбільш природною ML-задачею для цього датасету я вважаю **regression** — прогнозування вартості поїздки на основі інформації, доступної до її початку. Така модель могла б використовуватися для попередньої оцінки тарифу.